# 05 — Gold SISMEPRE
## Transformación Silver → Gold

Lee las 7 tablas Silver de SISMEPRE y las persiste directamente en Gold como Parquet.
Son tablas planas (no dimensionales), listas para análisis directo.

**Fuente:** `data/silver/sismepre/{tabla}/`  
**Destino:** `data/gold/sismepre_{tabla}/`


In [1]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import SILVER, GOLD, str_path, ensure_dirs
from src.spark_utils import get_spark, write_parquet
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

Imports OK


In [2]:
# Mapeo: tabla Silver → clave Gold
TABLE_MAP = {
    "rentas_formulario":       "sismepre_formulario",
    "rentas_preguntas":        "sismepre_preguntas",
    "rentas_respuestas":       "sismepre_respuestas",
    "rentas_estadistica":      "sismepre_estadistica",
    "rentas_esat_estadistica_atm": "sismepre_esat_atm",
    "rentas_entidad_estado":   "sismepre_entidad_estado",
    "rentas_ano_aplicacion":   "sismepre_ano_aplicacion",
}

SILVER_BASE = SILVER["sismepre"]
print(f"Silver base: {SILVER_BASE}")
ensure_dirs()

Silver base: /workspace/data/silver/sismepre
[OK] Estructura de directorios verificada en /workspace/data


In [3]:
spark = get_spark(app_name="MEF_Gold_SISMEPRE", memory="4g")

[OK] SparkSession lista | version=3.5.0 | master=local[*]


In [4]:
control = ControlManager(pipeline_name="gold_sismepre")
execution = control.start(input_parameters={"tables": list(TABLE_MAP.keys())})

metrics = {}
start_total = time.time()

try:
    for silver_table, gold_key in TABLE_MAP.items():
        print(f"\n{'─'*60}")
        print(f"{silver_table} → gold/{gold_key}")

        silver_path = str_path(SILVER_BASE / silver_table)
        gold_path   = str_path(GOLD[gold_key])

        silver_dir = SILVER_BASE / silver_table
        if not silver_dir.exists() or not any(silver_dir.rglob("*.parquet")):
            print(f"   Sin datos Silver — saltando (ejecutar 02_silver_sismepre primero)")
            metrics[gold_key] = 0
            continue

        df = spark.read.parquet(silver_path)
        n = write_parquet(df, gold_path, mode="overwrite", coalesce_to=1)
        metrics[gold_key] = n

    elapsed = time.time() - start_total
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={**metrics, "elapsed_sec": round(elapsed, 1)}
    )
    print(f"\nGold SISMEPRE completado en {elapsed:.1f}s")

except Exception as e:
    control.log_error("GoldSismepreError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

2026-06-19 11:04:54 | INFO     | mef_dw.audit.control_manager | [AUDIT] Ejecución iniciada | pipeline=gold_sismepre id=1920b0ab

────────────────────────────────────────────────────────────
rentas_formulario → gold/sismepre_formulario
  ✅ 98 filas → /workspace/data/gold/sismepre_formulario

────────────────────────────────────────────────────────────
rentas_preguntas → gold/sismepre_preguntas
  ✅ 836 filas → /workspace/data/gold/sismepre_preguntas

────────────────────────────────────────────────────────────
rentas_respuestas → gold/sismepre_respuestas
  ✅ 174,210 filas → /workspace/data/gold/sismepre_respuestas

────────────────────────────────────────────────────────────
rentas_estadistica → gold/sismepre_estadistica
  ✅ 233 filas → /workspace/data/gold/sismepre_estadistica

────────────────────────────────────────────────────────────
rentas_esat_estadistica_atm → gold/sismepre_esat_atm
  ✅ 134,144 filas → /workspace/data/gold/sismepre_esat_atm

──────────────────────────────────────

In [5]:
print("\nResumen Gold SISMEPRE:")
print(f"  {'Tabla Gold':<35} {'Filas':>10}")
print("  " + "-" * 47)
for k, n in metrics.items():
    status = "" if n > 0 else ""
    print(f"  {status} {k:<33} {n:>10,}")


Resumen Gold SISMEPRE:
  Tabla Gold                               Filas
  -----------------------------------------------
   sismepre_formulario                       98
   sismepre_preguntas                       836
   sismepre_respuestas                  174,210
   sismepre_estadistica                     233
   sismepre_esat_atm                    134,144
   sismepre_entidad_estado               19,037
   sismepre_ano_aplicacion                   26
